In [20]:
import pandas as pd
import numpy as np
import scipy.stats as sps

# ╔══════════════════════════════════════════════════════════╗
# ║        RENSEIGNER LES CHEMINS DES FICHIERS ICI           ║
# ╚══════════════════════════════════════════════════════════╝
PATH_SPOT = "spot_rates_98-24.xls"       # Fichier spots Datastream
PATH_PPP  = "OECD1960PPP.csv"            # Fichier PPP OCDE

# ── 1. Fonctions d'import et de calcul ──────────────────────────────────────

def spot_import():
    """Importe les taux de change spots depuis Datastream."""
    spot1 = pd.read_excel(PATH_SPOT, sheet_name="daily_spot_1", skiprows=1, index_col="Code")
    spot2 = pd.read_excel(PATH_SPOT, skiprows=1, sheet_name="daily_spot_2", index_col="Code")
    return spot1.join(spot2).filter(regex="ER")

def spt_dol_import():
    """Convertit tous les taux en cotation directe USD et nettoie les NaN."""
    spt = spot_import()
    spt_dol = 1 / (spt.drop(columns=["USDOLLR(ER)", "PHILPES(ER)", "THABAHT(ER)", "CROATKN(ER)", "BULGLEV(ER)"]).mul(1 / spt["USDOLLR(ER)"], axis=0))
    spt_dol["GBP(ER)"]    = spt["USDOLLR(ER)"]   # GBP déjà coté en USD
    spt_dol["USEURSP(ER)"] = spt["USEURSP(ER)"]  # EUR déjà coté en USD, on réintègre
    return spt_dol

def PPP_import():
    """Importe les niveaux de prix relatifs PPP depuis l'OCDE."""
    df_PPP = pd.read_csv(PATH_PPP, sep=",")
    df_PPP = df_PPP[["TIME_PERIOD", "CURRENCY", "OBS_VALUE"]]
    df_PPP = df_PPP.pivot_table(index="TIME_PERIOD", columns="CURRENCY")
    df_PPP.columns = df_PPP.columns.droplevel(0)
    df_PPP.index   = pd.to_datetime(df_PPP.index, format='%Y')
    return df_PPP

def weightsPPPstrat(RERdf, n_long=3):
    """Calcule les poids de la stratégie PPP (long sous-évalué, short surévalué)."""
    rebal_dates = RERdf.index
    long       = RERdf.apply(lambda r: r.nsmallest(n_long).index, axis=1)
    short      = RERdf.apply(lambda r: r.nlargest(n_long).index,  axis=1)

    weights = pd.DataFrame(0.0, index=rebal_dates, columns=RERdf.columns)
    for d in rebal_dates:
        weights.loc[d, long.loc[d]]  =  1 / n_long
        weights.loc[d, short.loc[d]] = -1 / n_long

    # Le shift(1) permet d'appliquer les poids calculés le jour J aux rendements de la période suivante
    return weights.shift(1)

def sharpe(ret, freq=12):
    return freq * ret.mean() / (ret.std() * np.sqrt(freq))

def max_dd(ret):
    cum = np.exp(ret.cumsum())
    return (cum / cum.cummax() - 1).min()


# ── 2. Chargement et préparation des données ─────────────────────────────────
if __name__ == "__main__":
    print("Chargement des données...")
    mapping = {
        'AUSTDOL(ER)': "AUD", 'BRACRUZ(ER)': 'BRL', 'CNDOLLR(ER)': 'CAD', 'CHILPES(ER)': 'CLP',
        'USEURSP(ER)': "EUR", 'HUNFORT(ER)': "HUF", 'INDRUPE(ER)': "INR", 'INDORUP(ER)': "IDR",
        'ISRSHEK(ER)': "ILS", 'JAPAYEN(ER)': "JPY", 'MEXPESO(ER)': "MXN", 'NZDOLLR(ER)': "NZD",
        'NORKRON(ER)': "NOK", 'POLZLOT(ER)': "PLN", 'CISRUBM(ER)': "RUB", 'SINGDOL(ER)': "SGD",
        'COMRAND(ER)': "ZAR", 'SWEKRON(ER)': "SEK", 'SWISSFR(ER)': "CHF", 'CZECHCM(ER)': "CZK", 'GBP(ER)': "GBP"
    }

    spt_dol = spt_dol_import().rename(columns=mapping)
    df_PPP  = PPP_import()

    G10 = ["AUD", "CAD", "CHF", "EUR", "GBP", "JPY", "NOK", "NZD", "SEK"]

    # ── 3. Dates de rebalancement (Fin de mois) ──────────────────────────────
    # MODIFICATION ICI : On prend le dernier jour de cotation du mois avec .max()
    rebal_mth = spt_dol.index.to_series().groupby(spt_dol.index.to_period("M")).max()

    # Dates trimestrielles (fin janvier, fin avril, fin juillet, fin octobre)
    rebal_qtr = rebal_mth[rebal_mth.dt.month.isin([1, 4, 7, 10])]

    spt_G10m = spt_dol.reindex(rebal_mth, method="ffill")
    # Calcul des rendements "fin de mois à fin de mois"
    logret_G10m = np.log(spt_G10m) - np.log(spt_G10m.shift(1))

    # ── 4. Calcul du Signal PPP Pure ─────────────────────────────────────────
    # Signal calculé aux dates TRIMESTRIELLES sur tout l'historique
    df_PPPm = df_PPP.reindex(rebal_qtr, method="ffill")
    df_RERm = spt_dol.reindex(rebal_qtr, method="ffill") * df_PPPm

    # Restriction à l'univers G10
    df_RERG10m = df_RERm[G10]

    # Poids trimestriels
    weightsG10_qtr_ppp = weightsPPPstrat(df_RERG10m)

    # Forward-fill des poids sur l'index mensuel pour avoir les rendements mois par mois
    weightsG10m_ppp = weightsG10_qtr_ppp.reindex(rebal_mth).ffill()

    # Rendements MENSUELS avec poids à rebalancement TRIMESTRIEL
    returns_ppp = (logret_G10m[G10] * weightsG10m_ppp).sum(axis=1)

    # ── 5. Métriques et Export ───────────────────────────────────────────────
    print("\n--- Performances de la Stratégie PPP Pure (Fin de mois) ---")
    print(f"Ratio de Sharpe       : {sharpe(returns_ppp):.4f}")
    print(f"Rendement annuel (%)  : {12 * returns_ppp.mean() * 100:.2f}")
    print(f"Volatilité ann. (%)   : {returns_ppp.std() * np.sqrt(12) * 100:.2f}")
    print(f"Maximum Drawdown (%)  : {max_dd(returns_ppp) * 100:.2f}")
    print(f"Skewness              : {sps.skew(returns_ppp.dropna()):.4f}")

    # Export Excel
    output_path = "PPP_Pure_Strategy_Returns_EOM.xlsx"
    ret_full_ppp = returns_ppp.rename("Monthly_Log_Return")

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        ret_full_ppp.to_frame().to_excel(writer, sheet_name="PPP_Pure_Full")

    print(f"\n✅ Les rendements fin-de-mois de la stratégie ont été exportés dans : {output_path}")

Chargement des données...

--- Performances de la Stratégie PPP Pure (Fin de mois) ---
Ratio de Sharpe       : 0.3404
Rendement annuel (%)  : 2.13
Volatilité ann. (%)   : 6.26
Maximum Drawdown (%)  : -13.89
Skewness              : 0.0280

✅ Les rendements fin-de-mois de la stratégie ont été exportés dans : PPP_Pure_Strategy_Returns_EOM.xlsx
